# 御厨こども食堂の謎：Colab試作

1. Googleアカウントでログインし、ランタイムは **CPU** を選びます。
2. 「調査環境を準備する」の左にある ▶ を押し、準備完了まで待ちます。
3. 上から順にセルを実行し、会話と設問を読み進めます。
4. 設問のセルでは、回答欄の上に問われている内容が表示されます。実行するとヒントを表示し、回答欄に入力してから実行すると判定します。
5. 調査には「＋コード」でセルを追加し、自分でコマンドを入力します。シェルコマンドの先頭には `!` を付けます。
6. 正解すると続くセルを読めるようになります。先に実行して表示されなかったセルは、正解後に再実行してください。

標的環境は寿司屋のWeb・Sambaを起動する試作です。後続章のWebメールや政府側サービスのColab対応は準備中です。

準備セルで、固定した版の教材をGitHubから取得し、必要なツールを導入します。


In [ ]:
# @title 調査環境を準備する
"""固定したGitHubコミット、またはNotebookに埋め込んだZIPからColabの教材・実行用ファイルを準備する。"""

from __future__ import annotations

import base64
import importlib.util
import io
import json
import re
import shutil
import stat
import tempfile
import zipfile
from pathlib import Path, PurePosixPath
from typing import Any
from urllib.error import URLError
from urllib.request import urlopen

SOURCE_FILES = (
    "docker/notebook/app/block_variants.py",
    "colab/colab_lab.py",
    "colab/colab_render.py",
    "colab/colab_web.py",
    "colab/colab_targets.py",
    "colab/checkpoints.json",
    "docker/sushi/samba/smb.conf",
    "docker/sushi/nginx.conf",
    "docker/portal/nginx.conf",
    "docker/portal/samba/smb.conf",
)
SOURCE_DIRS = (
    "docker/notebook/app/data",
    "docker/notebook/app/static/images",
    "docker/sushi/site",
    "docker/sushi/samba/share",
    "docker/sushi/webmail",
    "docker/portal/app",
    "docker/portal/samba/handover",
    "docker/portal/samba/share",
)
# 取得元のファイルとランタイム上の配置先。Docker版の設定はColab用に変換して使う。
STAGED_FILES = (
    ("docker/sushi/samba/smb.conf", "smb.conf.in"),
    ("docker/portal/samba/smb.conf", "smb-portal.conf.in"),
    ("docker/sushi/nginx.conf", "nginx-sushi.conf.in"),
    ("docker/portal/nginx.conf", "nginx-portal.conf.in"),
)
STAGED_DIRS = (
    ("docker/sushi/site", "site"),
    ("docker/sushi/samba/share", "share"),
    ("docker/sushi/webmail", "webmail"),
    ("docker/portal/app", "portal/app"),
    ("docker/portal/samba/handover", "portal-handover"),
    ("docker/portal/samba/share", "portal-handoff"),
    ("docker/notebook/app/static/images", "images"),
)
RUNTIME_MODULES = (
    "colab_lab.py",
    "colab_render.py",
    "colab_web.py",
    "colab_targets.py",
)


def source_files(source: Path) -> list[Path]:
    return [Path(name) for name in SOURCE_FILES] + [
        file.relative_to(source)
        for directory in SOURCE_DIRS
        for file in sorted((source / directory).rglob("*"))
        if file.is_file() and "__pycache__" not in file.parts
    ]


def load_lesson(source: Path) -> dict[str, Any]:
    spec = importlib.util.spec_from_file_location(
        "block_variants", source / "docker/notebook/app/block_variants.py"
    )
    if spec is None or spec.loader is None:
        raise ValueError("ブロックの環境差分処理を読み込めません。")
    variants = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(variants)
    data = source / "docker/notebook/app/data"
    manifest = json.loads((data / "chapters/manifest.json").read_text(encoding="utf-8"))
    chapters = [
        json.loads((data / "chapters" / filename).read_text(encoding="utf-8"))[
            "chapter"
        ]
        for filename in manifest["chapter_files"]
    ]
    chapters = variants.resolve_environment(chapters, "$colab")
    answers = json.loads((data / "answers.json").read_text(encoding="utf-8"))["answers"]
    speakers = json.loads(
        (data / "dialogue-speakers.json").read_text(encoding="utf-8")
    )["speakers"]
    # 合言葉の区切りはColab版だけで使うため、章JSONとは別に管理する。
    checkpoints = json.loads(
        (source / "colab/checkpoints.json").read_text(encoding="utf-8")
    )["checkpoints"]
    return {
        "title": "御厨こども食堂の謎",
        "chapters": chapters,
        "speakers": speakers,
        "checkpoints": checkpoints,
        "questions": [
            {**question, **answers[question["id"]]}
            for chapter in chapters
            for block in chapter["blocks"]
            if block["type"] == "questions"
            for question in block["items"]
        ],
    }


def prepare_from_source(source: Path, root: Path) -> None:
    lesson = load_lesson(source)
    credential = next(
        question
        for question in lesson["questions"]
        if question["id"] == "ch1_samba_credentials"
    )
    username, password = credential["accepted"][0].split("\t")
    # 必要ファイルがそろうまで、使用中の教材と進捗には触れない。
    with tempfile.TemporaryDirectory() as temporary:
        staging = Path(temporary)
        (staging / "lesson.json").write_text(
            json.dumps(lesson, ensure_ascii=False), encoding="utf-8"
        )
        (staging / "account.json").write_text(
            json.dumps({"username": username, "password": password}), encoding="utf-8"
        )
        for name in RUNTIME_MODULES:
            shutil.copyfile(source / "colab" / name, staging / name)
        for file, destination in STAGED_FILES:
            shutil.copyfile(source / file, staging / destination)
        for directory, destination in STAGED_DIRS:
            shutil.copytree(
                source / directory,
                staging / destination,
                ignore=shutil.ignore_patterns("__pycache__"),
            )
        root.mkdir(parents=True, exist_ok=True)
        for asset in staging.iterdir():
            destination = root / asset.name
            if destination.is_symlink() or destination.is_file():
                destination.unlink()
            elif destination.is_dir():
                shutil.rmtree(destination)
            shutil.move(str(asset), str(destination))


def fetch_archive(repository: str, commit: str) -> bytes:
    if not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError("取得先は40桁のコミットSHAで固定してください。")
    if not re.fullmatch(
        r"[A-Za-z0-9_-][A-Za-z0-9_.-]*/[A-Za-z0-9_-][A-Za-z0-9_.-]*", repository
    ):
        raise ValueError("GitHubリポジトリの指定が不正です。")
    url = f"https://codeload.github.com/{repository}/zip/{commit}"
    try:
        with urlopen(url, timeout=45) as response:
            return response.read()
    except (URLError, TimeoutError) as error:
        raise RuntimeError(
            "GitHubから教材を取得できませんでした。接続を確認し、準備セルを再実行してください。"
        ) from error


def install_archive(root: Path, data: bytes) -> None:
    """先頭に1階層を持つ教材ZIPから、取得対象だけを検証して配置する。"""
    with (
        tempfile.TemporaryDirectory() as temporary,
        zipfile.ZipFile(io.BytesIO(data)) as archive,
    ):
        source = Path(temporary)
        for member in archive.infolist():
            path = PurePosixPath(member.filename)
            if path.is_absolute() or ".." in path.parts:
                raise ValueError("教材ZIPに不正なパスが含まれています。")
            if member.is_dir():
                continue
            relative = PurePosixPath(*path.parts[1:])
            name = relative.as_posix()
            if name not in SOURCE_FILES and not any(
                name.startswith(directory + "/") for directory in SOURCE_DIRS
            ):
                continue
            if stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError(
                    "教材ZIPの展開対象にシンボリックリンクが含まれています。"
                )
            destination = source / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(archive.read(member))
        prepare_from_source(source, root)


def bootstrap_runtime(root: Path, repository: str, commit: str) -> None:
    install_archive(root, fetch_archive(repository, commit))


def bootstrap_embedded(root: Path, encoded: str) -> None:
    """Notebookに埋め込んだ教材ZIPから、通信せずに準備する。"""
    install_archive(root, base64.b64decode(encoded, validate=True))

SOURCE_COMMIT = 'cc7eeb9e815186f3f1bc3b4456cc502cc55222a5'
REPOSITORY = 'FakeDocument/MysteryOfMikuriyaKodomoShokudo'
import importlib, sys
ROOT = Path("/content/mikuriya-colab")
bootstrap_runtime(ROOT, REPOSITORY, SOURCE_COMMIT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import colab_web
importlib.reload(colab_web)
import colab_render
importlib.reload(colab_render)
import colab_lab
importlib.reload(colab_lab)
lab = colab_lab.Lab(ROOT)
lab.install_tools()
lab.start()


## 調査の中断・再開

章の区切りまで進むと合言葉が表示されます。
同じランタイムなら、準備セルを再実行しても正解済みの設問を引き継げます。
ランタイムが削除されたときは、準備セルを実行してから、次のセルに合言葉を入力して実行します。
合言葉を忘れたときは、空欄のまま実行すると、いまの進捗の合言葉を確認できます。


In [ ]:
# @title 合言葉で再開する
# @markdown 区切りで表示された合言葉をひらがなで入力します。
合言葉 = "" # @param {type:"string"}
_ = lab.resume(合言葉)


## 依頼書


In [ ]:
# @title 事務所の昼下がり
lab.show_block('prologue', 'opening')


In [ ]:
# @title 届いた依頼書
lab.show_block('prologue', 'request-letter')


In [ ]:
# @title 所長との作戦会議
lab.show_block('prologue', 'initial-conversation')


In [ ]:
# @title 調査結果を書き込む（確認 01）
# @markdown この依頼書を書いた人物は？
# @markdown
# @markdown 回答（氏名を入力）
回答 = "" # @param {type:"string"}
_ = lab.ask('prologue_requester', {'answer': 回答})


In [ ]:
# @title 調査結果を書き込む（確認 02）
# @markdown 行方が分からなくなっている人物は？
# @markdown
# @markdown 回答（氏名を入力）
回答 = "" # @param {type:"string"}
_ = lab.ask('prologue_missing', {'answer': 回答})


## 店の表と裏


In [ ]:
# @title 所長との作戦会議
lab.show_block('chapter-1', 'initial-conversation')


In [ ]:
# @title 開いているポートを記録する（調査 01）
# @markdown Nmapの結果で最初に表示された `open` の行を記録してください。
# @markdown
# @markdown ポート番号（例: 80）
ポート番号 = "" # @param {type:"string"}
# @markdown サービス名（例: http）
サービス名 = "" # @param {type:"string"}
_ = lab.ask('ch1_web_port_service', {'port': ポート番号, 'service': サービス名})


In [ ]:
# @title 開いているポートを記録する（調査 02）
# @markdown Nmapの結果でもう一つ表示された `open` の行を記録してください。
# @markdown
# @markdown ポート番号（例: 445）
ポート番号 = "" # @param {type:"string"}
# @markdown サービス名（例: netbios-ssn）
サービス名 = "" # @param {type:"string"}
_ = lab.ask('ch1_samba_port_service', {'port': ポート番号, 'service': サービス名})


In [ ]:
# @title Sambaの鍵を探す
lab.show_block('chapter-1', 'after-service-scan')


In [ ]:
# @title 見つけたディレクトリを記録する（調査 03）
# @markdown DIRBの結果を寿司屋HPの公開メニューと見比べてください。メニューに載っておらず、Sambaのログイン情報につながる2つのディレクトリは？
# @markdown
# @markdown ディレクトリ 1（例: /example/）
ディレクトリ_1 = "" # @param {type:"string"}
# @markdown ディレクトリ 2（例: /sample/）
ディレクトリ_2 = "" # @param {type:"string"}
_ = lab.ask('ch1_hidden_directories', {'first_directory': ディレクトリ_1, 'second_directory': ディレクトリ_2})


In [ ]:
# @title 二つのページを照合する
lab.show_block('chapter-1', 'after-hidden-pages')


In [ ]:
# @title Sambaの鍵を組み立てる（調査 04）
# @markdown 侵入できそうなアカウントのSamba認証情報を記録してください。
# @markdown
アカウントID = "" # @param {type:"string"}
# @markdown 初期パスワード（大文字・小文字を区別して入力）
初期パスワード = "" # @param {type:"string"}
_ = lab.ask('ch1_samba_credentials', {'account_id': アカウントID, 'password': 初期パスワード})


## 共有の奥


In [ ]:
# @title Sambaの入口を確かめる
lab.show_block('chapter-2', 'initial-conversation')


In [ ]:
# @title 共有の名前を記録する（調査 01）
# @markdown 一覧に表示された、調査に使えそうな共有の名前は？
# @markdown
# @markdown 回答（Sharename列の名前）
回答 = "" # @param {type:"string"}
_ = lab.ask('ch2_samba_share', {'answer': 回答})


In [ ]:
# @title staff共有へ接続する
lab.show_block('chapter-2', 'after-share-discovery')


In [ ]:
# @title 共有内の記録を調べる（調査 02）
# @markdown 共有内の資料を調べ、今回の失踪と似た過去の人物を記録してください。
# @markdown
# @markdown 回答（氏名を入力）
回答 = "" # @param {type:"string"}
_ = lab.ask('ch2_past_person', {'answer': 回答})


In [ ]:
# @title 店外とのつながりを調べる（調査 03）
# @markdown 共有内の資料から、店外との不自然なつながりを示す人物を記録してください。
# @markdown
# @markdown 回答（氏名を入力）
回答 = "" # @param {type:"string"}
_ = lab.ask('ch2_external_person', {'answer': 回答})


In [ ]:
# @title 次の調査先を組み立てる（調査 04）
# @markdown 共有内の資料を組み合わせ、次にアクセスする場所と、その入口で使うIDを記録してください。
# @markdown
# @markdown URL（http:// から入力）
URL = "" # @param {type:"string"}
# @markdown ログインID（確認したIDを入力）
ログインID = "" # @param {type:"string"}
_ = lab.ask('ch2_next_access', {'url': URL, 'login_id': ログインID})


## 政府への糸


In [ ]:
# @title 所長との作戦会議
lab.show_block('chapter-3', 'initial-conversation')


In [ ]:
# @title 検索結果を確認する（調査 01）
# @markdown 検索結果に現れた、片柳の業務メールアドレスを記録してください。
# @markdown
# @markdown 回答（メールアドレス）
回答 = "" # @param {type:"string"}
_ = lab.ask('ch3_contact_email', {'answer': 回答})


In [ ]:
# @title 片柳からのメール
lab.show_block('chapter-3', 'after-contact-email')


In [ ]:
# @title 連絡先を読み解く（調査 02）
# @markdown 片柳の業務メールアドレスから、所属先を示すドメインを記録してください。
# @markdown
# @markdown 回答（ドメイン名）
回答 = "" # @param {type:"string"}
_ = lab.ask('ch3_government_domain', {'answer': 回答})


## 海域保全対策室


In [ ]:
# @title それぞれの調査
lab.show_block('chapter-4', 'initial-conversation')


In [ ]:
# @title 入口を調べる（調査 01）
# @markdown 海域保全対策室のホストを調べ、2つのWebサービスとファイル共有サービスのポート番号を記録してください。
# @markdown
# @markdown 公開サイト（ポート番号）
公開サイト = "" # @param {type:"string"}
# @markdown もう一つのWebサイト（ポート番号）
もう一つのWebサイト = "" # @param {type:"string"}
# @markdown ファイル共有（ポート番号）
ファイル共有 = "" # @param {type:"string"}
_ = lab.ask('ch4_services', {'public_port': 公開サイト, 'catalog_port': もう一つのWebサイト, 'file_port': ファイル共有})


In [ ]:
# @title もう一つの入口（調査 02）
# @markdown 445番のファイル共有を調べ、文書目録に入るための閲覧コードを記録してください。まだログイン情報がなくても、共有一覧を確認できるでしょうか。
# @markdown
# @markdown 回答（閲覧コード）
回答 = "" # @param {type:"string"}
_ = lab.ask('ch4_catalog_entry', {'answer': 回答})


In [ ]:
# @title 担当範囲の向こう側（調査 03）
# @markdown 入手した閲覧コードで8080番の文書目録に入り、通常の照会範囲にない記録を調べてください。観測記録と古い伝承を照合し、そこに現れる存在の統一名称と、島が消える現象の呼び名を記録してください。
# @markdown
# @markdown 当室での統一名称（○○様）
当室での統一名称 = "" # @param {type:"string"}
# @markdown 島が消える伝承の呼び名（資料を照合して記録）
島が消える伝承の呼び名 = "" # @param {type:"string"}
_ = lab.ask('ch4_anomaly', {'name': 当室での統一名称, 'nature': 島が消える伝承の呼び名})


In [ ]:
# @title 守ろうとしているもの（調査 04）
# @markdown 観測記録と実施要領を、公開サイトの観測点一覧と照合してください。政府が守ろうとしている島と、維持しようとしている海域はどこですか。
# @markdown
# @markdown 守ろうとしている島（○○島）
守ろうとしている島 = "" # @param {type:"string"}
# @markdown 維持しようとしている海域（略称または正式名称）
維持しようとしている海域 = "" # @param {type:"string"}
_ = lab.ask('ch4_conservation_target', {'island': 守ろうとしている島, 'waters': 維持しようとしている海域})


In [ ]:
# @title 原簿のある場所へ（調査 05）
# @markdown 文書システムで得た情報を使い、政府側のファイル共有へ接続して原簿を取得してください。接続できた共有名と、9月分の受入照合簿に記載された受入番号を記録してください。
# @markdown
共有名 = "" # @param {type:"string"}
受入番号 = "" # @param {type:"string"}
_ = lab.ask('ch4_file_share', {'share': 共有名, 'intake_number': 受入番号})


In [ ]:
# @title 持ち出した原簿をつなぐ（調査 06）
# @markdown 第3章の御厨の連絡と受入照合簿を照合し、この原簿が笑真の引き渡しを扱うと確かめてください。確定配車表で予定日時と使用する入場口を確認し、受入口登録簿にあるその名称を記録してください。
# @markdown
# @markdown 引き渡し予定日時（YYYY-MM-DDTHH:MM）
引き渡し予定日時 = "" # @param {type:"string"}
# @markdown 引き渡し場所（受入口名）（受入口登録簿にある名称）
引き渡し場所_受入口名 = "" # @param {type:"string"}
_ = lab.ask('ch4_handoff_schedule', {'handoff_time': 引き渡し予定日時, 'handoff_place': 引き渡し場所_受入口名})


In [ ]:
# @title 残された時刻
lab.show_block('chapter-4', 'chapter-close')


## 調査報告書


In [ ]:
# @title 所長からの電話
lab.show_block('finale', 'final-call')


In [ ]:
# @title 情報整理（最終確認 01）
# @markdown 子どもたちは何のために引き渡されるのか？
# @markdown
# @markdown 供物を捧げる相手（名称）
供物を捧げる相手 = "" # @param {type:"string"}
# @markdown 保全する島（島名）
保全する島 = "" # @param {type:"string"}
_ = lab.ask('finale_purpose', {'offering': 供物を捧げる相手, 'purpose': 保全する島})


In [ ]:
# @title 情報整理（最終確認 02）
# @markdown 御厨水産が狙い、連れ去った子どもたちにはどのような共通点があった？
# @markdown
# @markdown 家庭・交友関係（共通点）
家庭_交友関係 = "" # @param ["", "孤立している", "良好", "不明"]
# @markdown 肉付き（共通点）
肉付き = "" # @param ["", "貧相", "良い", "不明"]
_ = lab.ask('finale_selection', {'isolation': 家庭_交友関係, 'build': 肉付き})


In [ ]:
# @title 情報整理（最終確認 03）
# @markdown 今回、供物として選定されてしまった標的は誰か？
# @markdown
# @markdown 回答（氏名）
回答 = "" # @param {type:"string"}
_ = lab.ask('finale_target', {'answer': 回答})


In [ ]:
# @title 情報整理（最終確認 04）
# @markdown 笑真の引き渡しが行われる具体的な日時と場所は？
# @markdown
# @markdown 日時（YYYY-MM-DDTHH:MM）
日時 = "" # @param {type:"string"}
# @markdown 場所（受入口名）
場所 = "" # @param ["", "裏手の荷捌き所", "正面玄関", "漁港"]
_ = lab.ask('finale_handoff', {'datetime': 日時, 'place': 場所})


In [ ]:
# @title 完成した調査報告書
lab.show_block('finale', 'completed-report')


In [ ]:
# @title 深夜の画面
lab.show_block('finale', 'decision')


In [ ]:
# @title 荷捌き所の対峙
lab.show_block('finale', 'confrontation')


In [ ]:
# @title 神懸かりと檻
lab.show_block('finale', 'sealing')


In [ ]:
# @title 一週間後の覚醒
lab.show_block('finale', 'awakening')


In [ ]:
# @title 小さな依頼の終わり
lab.show_block('finale', 'return-to-office')


In [ ]:
# @title 美咲からの手紙
lab.show_block('finale', 'misaki-letter')


In [ ]:
# @title いつもの事務所
lab.show_block('finale', 'epilogue')
